**PASS: fresh worked-example execution.** All saved outputs below came from this notebook's actual code. Run UTC: `2026-11-05T07:19:57.107965+00:00`; executed 6/6 code cells; 6 output objects; 0 saved errors. No execution errors are allowed or suppressed. These are provided worked examples; unanswered learner practice is separate Markdown. The synthetic scope still applies.
<!-- canonical-execution-status:end -->

# Tier 1 — completed worked exercise examples

**Complete reference solutions.** The historical filename `Tier1_Student.ipynb` is retained for existing links. This notebook and its local implementation are now fully worked examples, with no executable blank tasks. They are suitable for self-study and inspection; they are not an independent student submission.

For a dedicated presentation route, open `notebooks/Tier1_Instructor.ipynb` from the package. Independent prompts are in `teaching/STUDENT_PRACTICE.md`.

## What you will learn and why

A generator must use available information, assign probabilities and produce ordered output. These calculations connect that idea to code you can inspect. You will calculate one attention row, trace rectangular tensor shapes, separate masking from target shifting, turn likelihood into loss, and compare local with complete-sequence decisions.

Prerequisites: a short dot product, rows/columns of a matrix, probability and basic Python indexing. Budget for guided study: 60–90 minutes. Predict each result before running the following cell. The expected values are explained here because this is a completed worked example, not an independent assessment.

In [1]:
from pathlib import Path
import importlib.util
import inspect
import math
import torch
import torch.nn.functional as F

# Find this package from either notebooks/, student_exercises/, or package root.
package_candidates = [Path.cwd(), *Path.cwd().parents]
PACKAGE_ROOT = next(
    candidate for candidate in package_candidates
    if (candidate / "src/vgu_transformers.py").is_file()
    and (candidate / "notebooks/student_exercises/student_impl.py").is_file()
)
implementation_path = PACKAGE_ROOT / "notebooks/student_exercises/student_impl.py"
module_spec = importlib.util.spec_from_file_location("vgu_completed_tier12", implementation_path)
work = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(work)
assert Path(work.__file__).resolve() == implementation_path.resolve()
torch.manual_seed(9)
torch.set_num_threads(1)
torch.set_printoptions(precision=4, sci_mode=False)
print("Completed implementation:", implementation_path.relative_to(PACKAGE_ROOT).as_posix())
print("PyTorch:", torch.__version__)


Completed implementation: notebooks/student_exercises/student_impl.py
PyTorch: 2.10.0+cpu


## 1A — Attention: compare, normalize, mix

A query describes the information a prediction needs. Keys determine relevance; values carry the information to mix. These small vectors stand for outputs of learned projections:

$$q=[1,0],\quad K=\begin{bmatrix}1&0\\0&1\end{bmatrix},\quad V=\begin{bmatrix}2&0\\0&4\end{bmatrix}.$$

Dot products are $[1,0]$. The key width is two, so scaled scores are $[1/\sqrt{2},0]$. Softmax gives approximately $[0.6698,0.3302]$. Multiply each **whole value vector** by its weight and add: $0.6698[2,0]+0.3302[0,4]\approx[1.3395,1.3210]$.

The weights range over key positions. The output coordinates are information features, not next-token probabilities. Ask: if only V changes, which earlier quantities remain unchanged? Q/K comparisons and their weights.

In [2]:
q = torch.tensor([[1.0, 0.0]])
k = torch.eye(2)
v = torch.tensor([[2.0, 0.0], [0.0, 4.0]])
raw_scores = q @ k.transpose(-2, -1)
scaled_scores = raw_scores / math.sqrt(q.shape[-1])
output, weights = work.attention(q, k, v)
torch.testing.assert_close(weights.sum(-1), torch.ones(1))
torch.testing.assert_close(output, torch.tensor([[1.3395231, 1.3209538]]))
print("Raw scores:", raw_scores)
print("Scaled scores:", scaled_scores)
print("Weights over keys:", weights)
print("Weighted value output:", output)


Raw scores: tensor([[1., 0.]])
Scaled scores: tensor([[0.7071, 0.0000]])
Weights over keys: tensor([[0.6698, 0.3302]])
Weighted value output: tensor([[1.3395, 1.3210]])


## 1B — Shapes expose the computation

Use distinct axis sizes: batch $B=2$, heads $H=3$, queries $T=5$, source positions $S=7$, key width $d_k=4$, value width $d_v=6$.

| Tensor | Shape | Meaning |
|---|---|---|
| Q | `[2,3,5,4]` | One query per target position/head |
| K | `[2,3,7,4]` | Seven possible information sources |
| V | `[2,3,7,6]` | Six value features at each source |
| Scores and weights | `[2,3,5,7]` | Each query compares with seven keys |
| Output | `[2,3,5,6]` | One mixed value vector for each query |

Normalize over the last, key-position axis. The seven source positions disappear when we sum weighted values; the five queries remain. Rectangular examples make a wrong transpose easier to detect.

In [3]:
g = torch.Generator().manual_seed(301)
q = torch.randn(2, 3, 5, 4, generator=g)
k = torch.randn(2, 3, 7, 4, generator=g)
v = torch.randn(2, 3, 7, 6, generator=g)
output, weights = work.attention(q, k, v)
reference = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0)
torch.testing.assert_close(output, reference, atol=2e-6, rtol=2e-6)
assert weights.shape == (2, 3, 5, 7) and output.shape == (2, 3, 5, 6)
print("Q / K / V:", tuple(q.shape), tuple(k.shape), tuple(v.shape))
print("Score / weight shape:", tuple(weights.shape))
print("Output shape:", tuple(output.shape))
print("Largest difference from independent PyTorch API:", float((output-reference).abs().max()))


Q / K / V: (2, 3, 5, 4) (2, 3, 7, 4) (2, 3, 7, 6)
Score / weight shape: (2, 3, 5, 7)
Output shape: (2, 3, 5, 6)
Largest difference from independent PyTorch API: 2.384185791015625e-07


## 1C — Mask and shift solve different problems

The local vocabulary uses `I=0`, `love=1`, `AI=2`, `BOS=4`, `EOS=5`, `PAD=6`. Shift `[BOS,I,love,AI,EOS]` into decoder input `[BOS,I,love,AI]` and target `[I,love,AI,EOS]`.

At input I, the next-token label is love. A future input already contains love, so it must be hidden. Shifting chooses the label; a causal mask controls input visibility.

The additive mask contains **0 or −∞ before softmax**. Attention weights come **after softmax**. With equal unmasked scores, query row 1 has mask `[0,0,-inf,-inf]` and weights `[0.5,0.5,0,0]`. Never combine mask values and weights into one ambiguous cell.

In [4]:
mask = work.causal_mask(4)
ids = torch.tensor([[4, 0, 1, 2, 5]])
inputs, labels = work.next_token_pairs(ids)
assert inputs.tolist() == [[4, 0, 1, 2]]
assert labels.tolist() == [[0, 1, 2, 5]]
weights_for_equal_scores = torch.softmax(torch.zeros_like(mask) + mask, dim=-1)
torch.testing.assert_close(weights_for_equal_scores[1], torch.tensor([0.5, 0.5, 0.0, 0.0]))
print("Additive mask BEFORE softmax:\n", mask)
print("Weights AFTER softmax, equal scores:\n", weights_for_equal_scores)
print("Decoder input IDs:", inputs.tolist())
print("Next-token label IDs:", labels.tolist())
print("At input I, label love is predicted; future input love is blocked.")


Additive mask BEFORE softmax:
 tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])
Weights AFTER softmax, equal scores:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500]])
Decoder input IDs: [[4, 0, 1, 2]]
Next-token label IDs: [[0, 1, 2, 5]]
At input I, label love is predicted; future input love is blocked.


## 1D — Likelihood becomes a learning signal

For logits `[ln(2),0,0]`, exponentials are `[2,1,1]` and probabilities are `[0.5,0.25,0.25]`. If the correct target is class 2, negative log-likelihood is $-\log(0.25)=\log(4)$ and this one-token perplexity is $\exp(\log 4)=4$.

These are **vocabulary probabilities**, unlike attention weights over keys. The explicit `ignore_index=-100` says which labels would be excluded; none is excluded in this example. Lower teacher-forced loss alone does not establish free-running generation or held-out generalization.

In [5]:
logits = torch.tensor([[[math.log(2.0), 0.0, 0.0]]])
labels = torch.tensor([[2]])
probabilities = torch.softmax(logits, dim=-1)
loss = work.masked_cross_entropy(logits, labels, ignore_index=-100)
torch.testing.assert_close(probabilities, torch.tensor([[[0.5, 0.25, 0.25]]]))
assert abs(float(loss) - math.log(4.0)) < 1e-6
print("Vocabulary probabilities:", probabilities)
print("Correct target index:", labels.item())
print("Cross-entropy:", float(loss))
print("One-token perplexity:", float(loss.exp()))


Vocabulary probabilities: tensor([[[0.5000, 0.2500, 0.2500]]])
Correct target index: 2
Cross-entropy: 1.3862943649291992
One-token perplexity: 4.0


## 1E — A local best choice can lose globally

Use a fixed two-symbol tree, without EOS or a length penalty. `P(A)=0.6`, `P(B)=0.4`; after A, X/Y each have probability 0.5; after B, X has probability 1 and Y probability 0. Ties choose X.

Greedy commits to AX with joint probability 0.30. A width-3 search retains both initial prefixes and finds BX with probability 0.40. The code enumerates this tiny tree explicitly; the general generation functions are demonstrated in `02_Training_and_Generation_Worked.ipynb`.

This changes the search over one fixed model, not its probabilities. A higher model score need not improve task accuracy.

In [6]:
first_step = {"A": 0.6, "B": 0.4}
second_step = {"A": {"X": 0.5, "Y": 0.5}, "B": {"X": 1.0, "Y": 0.0}}
greedy_first = max(first_step, key=first_step.get)
greedy_second = max(second_step[greedy_first], key=second_step[greedy_first].get)
greedy_sequence = greedy_first + greedy_second
greedy_probability = first_step[greedy_first] * second_step[greedy_first][greedy_second]
beam_width = 3
retained_prefixes = sorted(first_step, key=lambda token: (-first_step[token], token))[:beam_width]
candidates = [(prefix + token, first_step[prefix] * probability)
              for prefix in retained_prefixes for token, probability in second_step[prefix].items()]
ranked = sorted(candidates, key=lambda item: (-item[1], item[0]))[:beam_width]
assert greedy_sequence == "AX" and abs(greedy_probability - 0.30) < 1e-9
assert ranked[0][0] == "BX" and abs(ranked[0][1] - 0.40) < 1e-9
print("Complete two-symbol candidates:", sorted(candidates))
print("Greedy:", greedy_sequence, greedy_probability)
print("Width-3 retained complete candidates:", ranked)


Complete two-symbol candidates: [('AX', 0.3), ('AY', 0.3), ('BX', 0.4), ('BY', 0.0)]
Greedy: AX 0.3
Width-3 retained complete candidates: [('BX', 0.4), ('AX', 0.3), ('AY', 0.3)]


## Transfer: explain what the code lets you check

For document transcription, image features provide source keys/values, previous output symbols provide the decoder prefix, and the model predicts the next symbol until EOS. The same shape, visibility, shift and loss checks still apply.

A small synthetic glyph exercise teaches the data flow; it does not establish real handwriting recognition or a need for useful target-history dependence. Start with a simple baseline and evaluate actual generated outputs on held-out cases.

For independent work, use the separate **STUDENT_PRACTICE.md** prompts and a new scratch file. These supplied calculations and passing checks are reference demonstrations, not evidence that a learner solved a fresh task unaided.